In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
    reasoning=False,
)

response = llm.invoke("Hello")
response

In [ ]:
# Wikipedia tool with retry handling

import json
import time

from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=4000,
    )
)


def wikipedia_invoke_with_retry(query, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            return wikipedia.invoke(query)
        except (json.JSONDecodeError, ConnectionError, TimeoutError) as error:
            if attempt == max_attempts - 1:
                return f"Wikipedia request failed after {max_attempts} attempts: {error}"
            time.sleep(2 ** attempt)


tool_response = wikipedia_invoke_with_retry("What is the capital of India?")
tool_response

In [ ]:
# Simple DuckDuckGo search

from langchain_community.tools import DuckDuckGoSearchRun

duckduckgo_search = DuckDuckGoSearchRun()

search_result = duckduckgo_search.invoke("Where is the Eiffel Tower located?")
print(search_result)

In [ ]:
# Creatin custome tools

from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def substarct(a: int, b: int) -> int:
    """Add two numbers."""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Add two numbers."""
    return a * b

print(add.invoke({"a":10, "b": 20}))  # Example usage of the custom tool



In [ ]:
tools = [wikipedia, add, substarct, multiply]

list_of_tools = {tool.name: tool for tool in tools}

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "20+10"
)

response.content

In [ ]:
# Optional: log in to Confident AI so evaluate() can publish results.
# The judge still comes from the setup cell (OpenAI or local Ollama).
import os
from pathlib import Path
from dotenv import load_dotenv
import deepeval

# Load CONFIDENT_API_KEY from notebooks/.env
cwd = Path.cwd()
env_candidates = [
    cwd / ".env",
    cwd / "notebooks" / ".env",
    cwd.parent / "notebooks" / ".env",
]
env_file = next((p for p in env_candidates if p.exists()), None)
if env_file is not None:
    load_dotenv(env_file, override=True)

api_key = os.getenv("CONFIDENT_API_KEY")
if not api_key or "paste_your_key" in api_key:
    print("Skipping Confident AI login. Set CONFIDENT_API_KEY in notebooks/.env to publish results.")
else:
    deepeval.login(api_key=api_key)
    print("Logged in to Confident AI.")


In [ ]:
# Create multiple set of datasets
# Create goldens from code and push to confidentAi
from deepeval.test_case import ToolCall

test_data = [
    {
        "input": "What is the Sum of 20+30",
        "expected_output": "50",
        "tool_called": [ToolCall(name="add")],
    }
]
test_data

In [ ]:
print(test_data[0]['input'])

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage


def query_agent(query: str) -> dict:
    messages = [HumanMessage(content=f"{query}. Use the appropriate tool.")]
    response = llm_with_tools.invoke(messages)

    if not response.tool_calls:
        return {
            "tool": None,
            "tool_input": None,
            "result": response.content,
        }

    tool_call = response.tool_calls[0]
    tool = list_of_tools[tool_call["name"]]
    tool_input = tool_call["args"]
    result = tool.invoke(tool_input)

    return {
        "tool": tool_call["name"],
        "tool_input": tool_input,
        "result": result,
    }


query_agent("10 + 20")

In [18]:
from deepeval import evaluate
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import LLMTestCase, ToolCall

agent_response = query_agent("10 + 20")

test_case = LLMTestCase(
    input="What is the Sum of 10+20?",
    expected_output="30",
    actual_output=str(agent_response["result"]),
    expected_tools=[
        ToolCall(
            name="add",
            input_parameters={"a": 10, "b": 20},
        )
    ],
    tools_called=[
        ToolCall(
            name=agent_response["tool"],
            input_parameters=agent_response["tool_input"],
        )
    ],
)

metric = ToolCorrectnessMetric()

results = evaluate(
    test_cases=[test_case],
    metrics=[metric],
)

results

✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=True)...

c:\Users\Girish Kulkarni\OneDrive\Documents\LLM_Testing\.venv\Lib\site-packages\rich\live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Tool Correctness           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=2872598;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=2872601;https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmubzx1zm001mpi0ta7ue7bbi\https://app.confident-ai.com/project/cmtymxcje003rpg0t5tl04m5y/test-runs/cmubzx1zm001mpi0ta7ue7bbi]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Tool Correctness', threshold=0.5, success=True, score=1.0, reason="[\n\t Tool Calling Reason: All expected tools ['add'] were called (order not considered).\n\t Tool Selection Reason: No available tools were provided to assess tool selection criteria\n]\n", strict_mode=False, flaky=False, evaluation_model=None, error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Expected Tools:\n[\n    ToolCall(\n        name="add",\n        type="FUNCTION",\n        input_parameters={\n            "a": 10,\n            "b": 20\n        }\n    )\n] \n \nTools Called:\n[\n    ToolCall(\n        name="add",\n        type="FUNCTION",\n        input_parameters={\n            "a": 10,\n            "b": 20\n        }\n    )\n] \n \nAvailable Tools: [] \n \nTool Selection Score: 1.0 \n \nTool Selection Reason: No available tools were provided to assess tool selection criteria'